# 06 - Personalized Recommendation Model
**Phase 2: Train personalized recommendation model**

### Approach: Personalized Popularity
Instead of ALS (which requires more cluster resources), we use a smarter baseline:
- Recommend popular items **within each customer's preferred department**
- Uses the `customer_preferred_department` feature from Phase 1
- Should beat the global popularity baseline!

### Goal: Beat the baseline!
| Model | MAP@12 |
|-------|--------|
| Popularity Baseline | 0.00299 |
| Personalized Popularity | ??? |

In [0]:
# ========================================
# SETUP & CONFIGURATION
# ========================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, date
import pandas as pd
import numpy as np
import mlflow
import mlflow.spark

# Storage Configuration
STORAGE_ACCOUNT = "ragadziyada"
STORAGE_KEY = "qIXjwo7EGK8an84BPCk446tqY9L7K7r3a9W2WB+voe2vUCvW1SK3qc/7UGOicKyBAtrptYcVfTB7+AStvWZi0A=="

PROCESSED_CONTAINER = "processed-p1"
CURATED_CONTAINER = "curated-p1"

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)

# Paths
TRANSACTIONS_CLEAN = f"abfss://{PROCESSED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/transactions_clean/"
CUSTOMER_FEATURES = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/customer_features/"
ARTICLE_FEATURES = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/article_features/"
MODEL_OUTPUT = f"abfss://{CURATED_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/models/"

# Settings
K = 12  # Top-K recommendations

print("✅ Configuration loaded!")

✅ Configuration loaded!


In [0]:
# ========================================
# MODEL SELECTION JUSTIFICATION
# ========================================

print("""
📋 MODEL SELECTION: Personalized Popularity

┌─────────────────────────────────────────────────────────────────────────┐
│ WHY NOT ALS (Alternating Least Squares)?                                │
├─────────────────────────────────────────────────────────────────────────┤
│ • ALS requires converting data to RDD format                            │
│ • Shared Databricks cluster blocks RDD operations (security policy)     │
│ • Even with 10% sampling (2.7M interactions), OOM errors occurred       │
│ • ALS needs significant memory for matrix factorization                 │
│ • Cluster limitation documented, not a design flaw                      │
└─────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────┐
│ WHY PERSONALIZED POPULARITY?                                            │
├─────────────────────────────────────────────────────────────────────────┤
│ • Leverages Phase 1 feature engineering (customer_preferred_department) │
│ • No RDD operations required - pure DataFrame/SQL                       │
│ • Scales to 1.3M customers without memory issues                        │
│ • Provides TRUE personalization (not same items for everyone)           │
│ • Interpretable: "recommend popular items in YOUR favorite department"  │
│ • Fast inference: simple lookup table join                              │
└─────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────┐
│ EXPECTED IMPROVEMENT OVER BASELINE                                      │
├─────────────────────────────────────────────────────────────────────────┤
│ • Baseline: Same 12 items for ALL customers (no personalization)        │
│ • Personalized: Top 12 from EACH customer's preferred department        │
│ • Hypothesis: Matching recommendations to shopping history improves     │
│   relevance and conversion                                              │
│ • Target: Beat baseline MAP@12 of 0.00299                               │
└─────────────────────────────────────────────────────────────────────────┘
""")


📋 MODEL SELECTION: Personalized Popularity

┌─────────────────────────────────────────────────────────────────────────┐
│ WHY NOT ALS (Alternating Least Squares)?                                │
├─────────────────────────────────────────────────────────────────────────┤
│ • ALS requires converting data to RDD format                            │
│ • Shared Databricks cluster blocks RDD operations (security policy)     │
│ • Even with 10% sampling (2.7M interactions), OOM errors occurred       │
│ • ALS needs significant memory for matrix factorization                 │
│ • Cluster limitation documented, not a design flaw                      │
└─────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────┐
│ WHY PERSONALIZED POPULARITY?                                            │
├─────────────────────────────────────────────────────────────────────────┤
│ • Leverages Phase 1 feature engineering 

In [0]:
# ========================================
# LOAD DATA & TRAIN/TEST SPLIT
# ========================================

# Load data
transactions_df = spark.read.parquet(TRANSACTIONS_CLEAN)
customer_features_df = spark.read.parquet(CUSTOMER_FEATURES)
article_features_df = spark.read.parquet(ARTICLE_FEATURES)

print(f"📦 Transactions: {transactions_df.count():,}")
print(f"📦 Customer features: {customer_features_df.count():,}")
print(f"📦 Article features: {article_features_df.count():,}")

# Temporal split (same as baseline notebook)
split_date = date(2020, 9, 15)

train_df = transactions_df.filter(F.col("t_dat") < F.lit(split_date))
test_df = transactions_df.filter(F.col("t_dat") >= F.lit(split_date))

print(f"\n📅 Temporal Split:")
print(f"   Train: {train_df.count():,} transactions")
print(f"   Test:  {test_df.count():,} transactions")

# Create ground truth
test_ground_truth = (
    test_df
    .groupBy("customer_id")
    .agg(F.collect_set("article_id").alias("actual_articles"))
)

print(f"\n✅ Test ground truth: {test_ground_truth.count():,} customers")

📦 Transactions: 28,813,419
📦 Customer features: 1,362,281
📦 Article features: 104,547

📅 Temporal Split:
   Train: 28,571,904 transactions
   Test:  241,515 transactions

✅ Test ground truth: 75,481 customers


In [0]:
# ========================================
# BUILD PERSONALIZED POPULARITY MODEL
# ========================================
# Calculate top-K popular articles PER DEPARTMENT

print("📊 Building Personalized Popularity Model...")
print("   Strategy: Recommend top items from customer's preferred department\n")

# Get article popularity by department (from TRAINING data only!)
article_dept_popularity = (
    train_df
    .join(
        article_features_df.select("article_id", "article_department_name"),
        on="article_id",
        how="left"
    )
    .groupBy("article_department_name", "article_id")
    .agg(F.count("*").alias("dept_sales"))
    .withColumn(
        "dept_rank",
        F.row_number().over(
            Window.partitionBy("article_department_name").orderBy(F.desc("dept_sales"))
        )
    )
    .filter(F.col("dept_rank") <= K)
)

num_depts = article_dept_popularity.select("article_department_name").distinct().count()
print(f"✅ Computed top-{K} articles for {num_depts} departments")

# Preview
display(article_dept_popularity.orderBy("article_department_name", "dept_rank").limit(15))

📊 Building Personalized Popularity Model...
   Strategy: Recommend top items from customer's preferred department

✅ Computed top-12 articles for 250 departments


article_department_name,article_id,dept_sales,dept_rank
AK Bottoms,0664871001,891,1
AK Bottoms,0717797001,487,2
AK Bottoms,0858692004,392,3
AK Bottoms,0708679002,376,4
AK Bottoms,0863502004,265,5
AK Bottoms,0863502005,237,6
AK Bottoms,0834069002,226,7
AK Bottoms,0858692003,218,8
AK Bottoms,0735740001,215,9
AK Bottoms,0749147002,213,10


In [0]:
# ========================================
# GENERATE PERSONALIZED RECOMMENDATIONS
# ========================================

# Get test customers with their preferred department
test_customers_with_prefs = (
    test_ground_truth
    .join(
        customer_features_df.select("customer_id", "customer_preferred_department"),
        on="customer_id",
        how="left"
    )
)

# Collect top-K articles per department as array
dept_recommendations = (
    article_dept_popularity
    .groupBy("article_department_name")
    .agg(F.collect_list("article_id").alias("dept_top_articles"))
)

# Join: each customer gets recommendations from their preferred department
personalized_predictions = (
    test_customers_with_prefs
    .join(
        dept_recommendations,
        test_customers_with_prefs.customer_preferred_department == dept_recommendations.article_department_name,
        how="left"
    )
    .select(
        "customer_id",
        "actual_articles",
        F.col("dept_top_articles").alias("predicted_articles"),
        "customer_preferred_department"
    )
)

# For customers with no preferred department or no match, use global top-K
global_top_k = (
    train_df
    .groupBy("article_id")
    .agg(F.count("*").alias("sales"))
    .orderBy(F.desc("sales"))
    .limit(K)
)
global_top_k_list = [row.article_id for row in global_top_k.collect()]

personalized_predictions = personalized_predictions.withColumn(
    "predicted_articles",
    F.when(
        F.col("predicted_articles").isNull(),
        F.array([F.lit(x) for x in global_top_k_list])
    ).otherwise(F.col("predicted_articles"))
)

pred_count = personalized_predictions.count()
print(f"✅ Generated personalized recommendations for {pred_count:,} customers")

display(personalized_predictions.select("customer_id", "customer_preferred_department", "predicted_articles").limit(5))

✅ Generated personalized recommendations for 75,481 customers


customer_id,customer_preferred_department,predicted_articles
64a9d684515aae15f66a71afe1bbdfa64c09e24222a19b47a5c49368d9a03e0d,Basic 1,"List(0717490008, 0759871001, 0711053003, 0624486001, 0806388002, 0806388001, 0683662005, 0806388003, 0772902001, 0715624001, 0189616006, 0717490010)"
e532e821002e6834df0d058b9b8cd688f49233c23ba1e65899a9a2f9ce419de5,Blouse,"List(0507909001, 0695632002, 0695632001, 0507910001, 0762846001, 0716672001, 0762846008, 0618800001, 0850917001, 0762846006, 0507909003, 0697054014)"
d83095facc09cff9fceab9811f2cf9a5c2d3ec4e21ffd5c17639908a928afc58,Casual Lingerie,"List(0464297007, 0723469001, 0719655001, 0719957006, 0351933001, 0465655010, 0506098007, 0723469002, 0557994003, 0704756001, 0464297021, 0757971001)"
25d5af127b8f8cae6fa7b1bf340b373976537bdba57532877c36e3c359611cb4,Denim Trousers,"List(0399223001, 0636323001, 0399223029, 0399223033, 0799365002, 0714790020, 0714790003, 0687704001, 0399087014, 0658030011, 0714790008, 0399087010)"
df771da11036adfb7b3f424a13a99388542f6511f20ea4db3d6ea59c69e0383c,Dress,"List(0745475001, 0745475002, 0817353008, 0817353002, 0740943001, 0857812004, 0685448004, 0702790004, 0685429004, 0863001001, 0685448005, 0743594003)"


In [0]:
# ========================================
# EVALUATE PERSONALIZED MODEL
# ========================================

def evaluate_recommendations(predictions_df, k=12):
    """Calculate MAP@K and Recall@K"""
    
    eval_pd = predictions_df.select("customer_id", "actual_articles", "predicted_articles").toPandas()
    
    def average_precision_at_k(actual, predicted, k):
        actual = list(actual) if actual is not None else []
        predicted = list(predicted) if predicted is not None else []
        if len(actual) == 0:
            return 0.0
        predicted = predicted[:k]
        score = 0.0
        num_hits = 0.0
        for i, p in enumerate(predicted):
            if p in actual and p not in predicted[:i]:
                num_hits += 1.0
                score += num_hits / (i + 1.0)
        return score / min(len(actual), k)
    
    def recall_at_k(actual, predicted, k):
        actual = list(actual) if actual is not None else []
        predicted = list(predicted) if predicted is not None else []
        if len(actual) == 0:
            return 0.0
        predicted = set(predicted[:k])
        actual = set(actual)
        return len(predicted & actual) / len(actual)
    
    eval_pd['ap'] = eval_pd.apply(
        lambda row: average_precision_at_k(row['actual_articles'], row['predicted_articles'], k), 
        axis=1
    )
    eval_pd['recall'] = eval_pd.apply(
        lambda row: recall_at_k(row['actual_articles'], row['predicted_articles'], k), 
        axis=1
    )
    
    return {
        'MAP@K': eval_pd['ap'].mean(),
        'Recall@K': eval_pd['recall'].mean(),
        'K': k,
        'num_customers': len(eval_pd)
    }

# Evaluate
print("🔄 Evaluating personalized model...")
metrics = evaluate_recommendations(personalized_predictions, k=K)

# Compare with baseline
baseline_map = 0.002989
baseline_recall = 0.008016

print("\n" + "=" * 60)
print("📊 MODEL COMPARISON")
print("=" * 60)
print(f"{'Model':<30} {'MAP@12':<15} {'Recall@12':<15}")
print("-" * 60)
print(f"{'Popularity Baseline':<30} {baseline_map:<15.6f} {baseline_recall:<15.6f}")
print(f"{'Personalized Popularity':<30} {metrics['MAP@K']:<15.6f} {metrics['Recall@K']:<15.6f}")
print("-" * 60)

# Calculate improvement
map_improvement = ((metrics['MAP@K'] - baseline_map) / baseline_map) * 100
recall_improvement = ((metrics['Recall@K'] - baseline_recall) / baseline_recall) * 100

print(f"\n📈 Improvement over baseline:")
print(f"   MAP@12:    {map_improvement:+.1f}%")
print(f"   Recall@12: {recall_improvement:+.1f}%")
print("=" * 60)

🔄 Evaluating personalized model...

📊 MODEL COMPARISON
Model                          MAP@12          Recall@12      
------------------------------------------------------------
Popularity Baseline            0.002989        0.008016       
Personalized Popularity        0.008065        0.020969       
------------------------------------------------------------

📈 Improvement over baseline:
   MAP@12:    +169.8%
   Recall@12: +161.6%


In [0]:
# ========================================
# ERROR ANALYSIS: Where Does the Model Fail?
# ========================================

print("🔍 ERROR ANALYSIS\n")

# 1. Customers with zero hits (no correct predictions)
zero_hits = personalized_predictions.filter(
    F.size(F.array_intersect("actual_articles", "predicted_articles")) == 0
)
zero_hit_count = zero_hits.count()
total_customers = personalized_predictions.count()

print(f"📊 Prediction Accuracy Breakdown:")
print(f"   Total test customers: {total_customers:,}")
print(f"   Customers with 0 correct predictions: {zero_hit_count:,} ({zero_hit_count/total_customers*100:.1f}%)")
print(f"   Customers with ≥1 correct prediction: {total_customers - zero_hit_count:,} ({(total_customers-zero_hit_count)/total_customers*100:.1f}%)")

# 2. Analyze failure cases
print("\n📋 FAILURE ANALYSIS:")
print("""
┌─────────────────────────────────────────────────────────────────────────┐
│ WHY MODEL FAILS (Common Patterns)                                       │
├─────────────────────────────────────────────────────────────────────────┤
│ 1. CROSS-DEPARTMENT PURCHASES                                           │
│    • Customer's preferred dept is "Dresses" but they buy "Shoes"        │
│    • Model only recommends from preferred department                    │
│    • Solution: Multi-department recommendations (future work)           │
├─────────────────────────────────────────────────────────────────────────┤
│ 2. COLD START FOR NEW ITEMS                                             │
│    • New articles in test period not in training popularity rankings    │
│    • Model can't recommend items it hasn't seen                         │
│    • Solution: Hybrid with content-based filtering (future work)        │
├─────────────────────────────────────────────────────────────────────────┤
│ 3. LONG-TAIL PREFERENCES                                                │
│    • Some customers buy niche items not in top-12 of any department     │
│    • Popularity-based models favor mainstream items                     │
│    • Solution: Collaborative filtering like ALS (needs more resources)  │
├─────────────────────────────────────────────────────────────────────────┤
│ 4. SEASONAL SHIFTS                                                      │
│    • Training on 2+ years, testing on last 7 days                       │
│    • Fashion trends change; old popular items may not be relevant       │
│    • Solution: Time-weighted popularity or recency features             │
└─────────────────────────────────────────────────────────────────────────┘
""")

# 3. Sample failed predictions
print("\n📋 Sample Failed Predictions (0 hits):")
failed_sample = zero_hits.select(
    "customer_id", 
    "customer_preferred_department",
    F.slice("actual_articles", 1, 3).alias("bought_sample"),
    F.slice("predicted_articles", 1, 3).alias("recommended_sample")
).limit(3)
display(failed_sample)

🔍 ERROR ANALYSIS

📊 Prediction Accuracy Breakdown:
   Total test customers: 75,481
   Customers with 0 correct predictions: 71,736 (95.0%)
   Customers with ≥1 correct prediction: 3,745 (5.0%)

📋 FAILURE ANALYSIS:

┌─────────────────────────────────────────────────────────────────────────┐
│ WHY MODEL FAILS (Common Patterns)                                       │
├─────────────────────────────────────────────────────────────────────────┤
│ 1. CROSS-DEPARTMENT PURCHASES                                           │
│    • Customer's preferred dept is "Dresses" but they buy "Shoes"        │
│    • Model only recommends from preferred department                    │
│    • Solution: Multi-department recommendations (future work)           │
├─────────────────────────────────────────────────────────────────────────┤
│ 2. COLD START FOR NEW ITEMS                                             │
│    • New articles in test period not in training popularity rankings    │
│    • Model can't recomm

customer_id,customer_preferred_department,bought_sample,recommended_sample
69db3dc87046d1720a80ec4c199744c33e49d90d9c68b49833a4c10c9f6d7ea4,Jacket Casual,List(0874110016),"List(0497640002, 0497640008, 0551504010)"
09c4eb393e488c46572c64e945e6173461c81fdbeb19c2afe361462d6c4d0602,Denim Trousers,"List(0914441004, 0867969006)","List(0399223001, 0636323001, 0399223029)"
21fb28d4c24d8382097cc1b715239742a8a8f3a407bebd1790ba32e3849c27a7,Denim Trousers,"List(0904415001, 0898886001, 0898886002)","List(0399223001, 0636323001, 0399223029)"


In [0]:
# ========================================
# LOG TO MLFLOW
# ========================================

experiment_name = "/Shared/hm_recommendation_60304248"
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name="personalized_popularity") as run:
    
    # Log parameters
    mlflow.log_param("model_type", "personalized_popularity")
    mlflow.log_param("k", K)
    mlflow.log_param("strategy", "department_based_recommendations")
    mlflow.log_param("train_end_date", str(split_date))
    mlflow.log_param("num_departments", num_depts)
    mlflow.log_param("test_customers", metrics['num_customers'])
    
    # Log metrics
    mlflow.log_metric("map_at_12", metrics['MAP@K'])
    mlflow.log_metric("recall_at_12", metrics['Recall@K'])
    mlflow.log_metric("map_improvement_pct", map_improvement)
    mlflow.log_metric("recall_improvement_pct", recall_improvement)
    
    # Get run ID
    personalized_run_id = run.info.run_id

print("✅ MLflow experiment logged!")
print(f"   Experiment: {experiment_name}")
print(f"   Run ID: {personalized_run_id}")

✅ MLflow experiment logged!
   Experiment: /Shared/hm_recommendation_60304248
   Run ID: 3bdc62ff0f5e4665a5c74ff66e8c9cc7


In [0]:
# ========================================
# SAVE MODEL RESULTS
# ========================================

# Save comparison results
model_comparison = spark.createDataFrame([
    {
        "model_name": "popularity_baseline",
        "map_at_12": 0.002989,
        "recall_at_12": 0.008016,
        "improvement_pct": 0.0,
        "run_date": datetime.now().isoformat()
    },
    {
        "model_name": "personalized_popularity",
        "map_at_12": float(metrics['MAP@K']),
        "recall_at_12": float(metrics['Recall@K']),
        "improvement_pct": float(map_improvement),
        "run_date": datetime.now().isoformat()
    }
])

model_comparison.write.mode("overwrite").parquet(f"{MODEL_OUTPUT}model_comparison/")

print("✅ Model comparison saved!")
print(f"   Location: {MODEL_OUTPUT}model_comparison/")
display(model_comparison)

✅ Model comparison saved!
   Location: abfss://curated-p1@ragadziyada.dfs.core.windows.net/models/model_comparison/


improvement_pct,map_at_12,model_name,recall_at_12,run_date
0.0,0.002989,popularity_baseline,0.008016,2026-04-10T10:44:24.730938
169.83135643781603,0.008065259243926322,personalized_popularity,0.020968812831916674,2026-04-10T10:44:24.730952


In [0]:
# ========================================
# FINAL MODEL COMPARISON
# ========================================

print("""
┌─────────────────────────────────────────────────────────────────────────┐
│                    MODEL COMPARISON SUMMARY                             │
├──────────────────────┬──────────────┬──────────────┬────────────────────┤
│ Model                │ MAP@12       │ Recall@12    │ vs Baseline        │
├──────────────────────┼──────────────┼──────────────┼────────────────────┤
│ Popularity Baseline  │ 0.00299      │ 0.00802      │ —                  │
│ Personalized Pop.    │ 0.00806      │ 0.02097      │ +169.8% (MAP)      │
└──────────────────────┴──────────────┴──────────────┴────────────────────┘

KEY INSIGHTS:
✅ Personalized model BEATS baseline by 170% on MAP@12
✅ Recall improved by 161% - more relevant items found
✅ Proves that department-based personalization adds value
✅ Simple, interpretable, and scalable approach

LIMITATIONS ACKNOWLEDGED:
⚠️ Cannot handle cross-department purchases
⚠️ Cold start problem for new items
⚠️ Static recommendations (no real-time updates)
""")


┌─────────────────────────────────────────────────────────────────────────┐
│                    MODEL COMPARISON SUMMARY                             │
├──────────────────────┬──────────────┬──────────────┬────────────────────┤
│ Model                │ MAP@12       │ Recall@12    │ vs Baseline        │
├──────────────────────┼──────────────┼──────────────┼────────────────────┤
│ Popularity Baseline  │ 0.00299      │ 0.00802      │ —                  │
│ Personalized Pop.    │ 0.00806      │ 0.02097      │ +169.8% (MAP)      │
└──────────────────────┴──────────────┴──────────────┴────────────────────┘

KEY INSIGHTS:
✅ Personalized model BEATS baseline by 170% on MAP@12
✅ Recall improved by 161% - more relevant items found
✅ Proves that department-based personalization adds value
✅ Simple, interpretable, and scalable approach

LIMITATIONS ACKNOWLEDGED:
⚠️ Cannot handle cross-department purchases
⚠️ Cold start problem for new items
⚠️ Static recommendations (no real-time updates)



# 📊 Notebook 06 Summary

## Model Comparison

| Model | MAP@12 | Recall@12 | Improvement |
|-------|--------|-----------|-------------|
| Popularity Baseline | 0.00299 | 0.00802 | — |
| Personalized Popularity | 0.00806 | 0.02097 | **+170%** |

## What We Did
1. ✅ Built **Personalized Popularity** model using Phase 1 features
2. ✅ Used `customer_preferred_department` to personalize recommendations
3. ✅ Evaluated on 75,481 test customers
4. ✅ Achieved **170% improvement** over baseline!
5. ✅ Logged to MLflow for tracking
6. ✅ Saved results to Azure Data Lake

## Why This Works
- Baseline: Same 12 items for everyone (no personalization)
- Personalized: Top 12 items **from each customer's preferred department**
- Customers get recommendations matching their shopping history!

## Next Steps
- **Notebook 07:** Model registration & versioning
- **Notebook 08:** Deployment to Azure
- **Notebook 09:** Deployment validation

---
*Phase 2 - Model Development | H&M Fashion Recommendations | Team 60304248*